# 02 · Honest Charts Lab: how charts lie, and how to fix them
**Presentation & Data Storytelling with AI Tools** · Day 1 · Module 2 · about 25 minutes

Each lab shows a chart that is **technically accurate but misleading**. Your job:
1. Run the cell and write down what the chart *seems* to say.
2. Find the trick.
3. Change `HONEST = False` to `HONEST = True` ✏️ and run it again.

The tricks are recreations of real cases that made the news (the numbers here are made up for practice and labeled that way). AI tools reproduce these tricks too, so you need to catch them yourself.

In [ ]:
# ▶ Tools + story style
import pandas as pd
import numpy as np

# ▶ STORY STYLE — run once. Every chart after this cell gets the clean look.
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

NAVY, CORAL, GRAY, LIGHT, INK = "#1B2A41", "#FF6F59", "#BFC5CF", "#EEF1F5", "#4A5568"
plt.rcParams.update({
    "figure.figsize": (10, 5.6), "figure.dpi": 110,
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.spines.left": False, "axes.grid": True, "axes.grid.axis": "y",
    "grid.color": LIGHT, "grid.linewidth": 1,
    "axes.edgecolor": GRAY, "axes.labelcolor": INK, "axes.axisbelow": True,
    "xtick.color": INK, "ytick.color": INK, "ytick.left": False,
    "font.size": 11, "axes.titlesize": 16, "axes.titleweight": "bold",
})

def add_titles(ax, title, subtitle=""):
    """Headline (the insight) + a smaller subtitle (what is measured), left-aligned."""
    ax.set_title(title, loc="left", pad=34 if subtitle else 12, color=NAVY)
    if subtitle:
        ax.text(0, 1.035, subtitle, transform=ax.transAxes, fontsize=11, color=INK, va="bottom")

def add_source(fig, text):
    """Small source line in the bottom-left corner: always say where data came from."""
    fig.text(0.01, -0.02, text, fontsize=9, color="#8A94A6", ha="left", va="top")

print("Story style loaded ✔")

## Lab 1 · The truncated axis
A team presents two plans' customer-satisfaction scores (made-up numbers).

In [ ]:
HONEST = False   # ✏️ change to True

plans = pd.DataFrame({"plan": ["Plan A", "Plan B"], "satisfaction": [82.1, 84.3]})
fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(plans["plan"], plans["satisfaction"], color=[GRAY, CORAL], width=0.55)
for x, v in zip(plans["plan"], plans["satisfaction"]):
    ax.text(x, v, f"{v}%", ha="center", va="bottom", fontweight="bold", color=NAVY)
if HONEST:
    ax.set_ylim(0, 100)
    add_titles(ax, "Plan B scores 2.2 points higher than Plan A", "Customer satisfaction, % (illustrative data)")
else:
    ax.set_ylim(81.5, 84.6)
    add_titles(ax, "Plan B crushes Plan A!", "Customer satisfaction, % (illustrative data)")
plt.show()

<details><summary><b>Reveal: what's the trick?</b></summary>

The y-axis starts at 81.5 instead of 0, so a 2.2-point difference looks like Plan B is several times bigger. **Bars encode size by length, so bar charts must start at zero.** (Line charts may zoom in, because they show change, not size; say so in the subtitle.)
</details>

## Lab 2 · The upside-down axis
In 2014 a news graphic about a Florida law flipped its y-axis (zero at the top) to look like dripping blood. Most readers saw deaths **falling** after the law, when the numbers showed them **rising**. This recreation uses made-up numbers with the same shape.

In [ ]:
HONEST = False   # ✏️ change to True

years = np.arange(1995, 2014)
rng = np.random.default_rng(7)
deaths = np.where(years < 2005, 520 - (years - 1995) * 8, 470 + (years - 2005) * 32) + rng.normal(0, 12, len(years))

fig, ax = plt.subplots()
ax.plot(years, deaths, color=CORAL, lw=3)
ax.axvline(2005, color=INK, ls="--", lw=1)
ax.text(2005.2, deaths.max(), "  2005: new law takes effect", color=INK, va="top")
if HONEST:
    ax.set_ylim(0, 900)
    add_titles(ax, "Deaths rose by about half in the eight years after the law", "Annual deaths (illustrative data)")
else:
    ax.set_ylim(900, 0)          # zero at the TOP: the trick
    ax.fill_between(years, deaths, 0, color=CORAL, alpha=0.25)
    add_titles(ax, "Deaths after the new law", "Annual deaths (illustrative data)")
plt.show()

<details><summary><b>Reveal</b></summary>

The axis runs from 0 at the top to 900 at the bottom. Our brains read "up = more," so the rising line looks like a decline. **Never invert an axis for style.** If a convention is unusual, label it loudly or don't use it.
</details>

## Lab 3 · Dates out of order
In May 2020 a state health department posted a bar chart where the dates on the x-axis were **not in calendar order** (they were sorted from highest to lowest). The chart seemed to show cases falling every day. It was taken down and corrected. Here is the same trick with made-up numbers.

In [ ]:
HONEST = False   # ✏️ change to True

days = pd.date_range("2025-04-20", periods=15, freq="D")
rng = np.random.default_rng(3)
cases = (60 + rng.normal(0, 9, len(days))).round().astype(int)
daily = pd.DataFrame({"date": days, "cases": cases})

shown = daily.sort_values("date") if HONEST else daily.sort_values("cases", ascending=False)
fig, ax = plt.subplots()
ax.bar(shown["date"].dt.strftime("%b %d"), shown["cases"], color=NAVY if HONEST else CORAL)
ax.tick_params(axis="x", rotation=60)
if HONEST:
    add_titles(ax, "New cases stayed flat at about 60 a day", "Daily new cases, in date order (illustrative data)")
else:
    add_titles(ax, "New cases are falling fast", "Daily new cases (illustrative data)")
plt.show()

<details><summary><b>Reveal</b></summary>

Read the x-axis labels: the dates jump around. Sorting by value manufactures a trend. **Time always runs left to right, in order, with even spacing.** When you ask an AI tool to "make the chart look better," check that it didn't re-sort your time axis.
</details>

## Lab 4 · The cherry-picked window
Same data, different start date, opposite story. ✏️ Change `START` and watch the headline you'd be tempted to write.

In [ ]:
START = "2025-03"   # ✏️ try "2020-01" to see the whole picture

months = pd.date_range("2020-01-01", "2025-12-01", freq="MS")
rng = np.random.default_rng(11)
trend = np.linspace(100, 62, len(months))
trend[-10:] = trend[-10:] + np.linspace(0, 14, 10)          # a short rebound at the end
sales = pd.Series(trend + rng.normal(0, 2.5, len(months)), index=months)

view = sales[START:]
change = (view.iloc[-1] / view.iloc[0] - 1) * 100
fig, ax = plt.subplots()
ax.plot(view.index, view.values, color=CORAL, lw=3)
add_titles(ax, f"Sales {'up' if change > 0 else 'down'} {abs(change):.0f}% since {view.index[0]:%b %Y}",
           "Monthly sales index (illustrative data)")
plt.show()

<details><summary><b>Reveal</b></summary>

From March 2025 sales look like they are surging; from 2020 they are down by about a quarter, with a small recent bounce. **Ask: why does the chart start here?** Honest storytellers show enough history for context, then zoom in and say why.
</details>

## Lab 5 · The dual axis that "proves" a link
Two unrelated made-up series, each on its own y-axis. The designer can stretch either axis until the lines kiss.

In [ ]:
HONEST = False   # ✏️ change to True

years = np.arange(2012, 2026)
rng = np.random.default_rng(42)
coffee = 40 + np.cumsum(rng.normal(1.2, 1.0, len(years)))      # "coffee shops in town"
scores = 610 + np.cumsum(rng.normal(2.0, 2.2, len(years)))     # "average test score"
r = np.corrcoef(coffee, scores)[0, 1]

if not HONEST:
    fig, ax = plt.subplots()
    ax.plot(years, coffee, color=CORAL, lw=3)
    ax2 = ax.twinx()
    ax2.plot(years, scores, color=NAVY, lw=3)
    ax.set_ylabel("Coffee shops", color=CORAL); ax2.set_ylabel("Avg test score", color=NAVY)
    add_titles(ax, "More coffee shops, smarter students?", "Illustrative data")
else:
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.8))
    a1.plot(years, coffee, color=CORAL, lw=3); a1.set_ylim(0, coffee.max() * 1.15)
    a2.plot(years, scores, color=NAVY, lw=3);  a2.set_ylim(0, scores.max() * 1.15)
    add_titles(a1, "Coffee shops", "count (illustrative)")
    add_titles(a2, "Average test score", "points (illustrative)")
    fig.suptitle(f"Both rose over time (r = {r:.2f}), but nothing here shows one causes the other",
                 x=0.01, ha="left", fontweight="bold", color=NAVY, y=1.08)
plt.show()

<details><summary><b>Reveal</b></summary>

Any two things that grow over time will correlate. Two y-axes let the designer pick scales that make the lines overlap. **Prefer two panels, each starting at zero, and never imply cause from a shared trend.** For fun examples, see Tyler Vigen's *Spurious Correlations* (tylervigen.com/spurious-correlations).
</details>

## Lab 6 · Anscombe's quartet: why you must always plot the data
In 1973 statistician Francis Anscombe built four tiny datasets with (almost) identical averages, variances and correlations. These are his real numbers. First look at the statistics, then at the pictures.

In [ ]:
x = [10, 8, 13, 9, 11, 14, 6, 4, 12, 7, 5]
quartet = {
    "I":   (x, [8.04, 6.95, 7.58, 8.81, 8.33, 9.96, 7.24, 4.26, 10.84, 4.82, 5.68]),
    "II":  (x, [9.14, 8.14, 8.74, 8.77, 9.26, 8.10, 6.13, 3.10, 9.13, 7.26, 4.74]),
    "III": (x, [7.46, 6.77, 12.74, 7.11, 7.81, 8.84, 6.08, 5.39, 8.15, 6.42, 5.73]),
    "IV":  ([8, 8, 8, 8, 8, 8, 8, 19, 8, 8, 8], [6.58, 5.76, 7.71, 8.84, 8.47, 7.04, 5.25, 12.50, 5.56, 7.91, 6.89]),
}
stats = pd.DataFrame({
    name: {"mean x": np.mean(xs), "mean y": np.mean(ys), "variance y": np.var(ys, ddof=1),
           "correlation": np.corrcoef(xs, ys)[0, 1]}
    for name, (xs, ys) in quartet.items()
}).T.round(2)
stats

In [ ]:
fig, axes = plt.subplots(1, 4, figsize=(15, 4), sharex=True, sharey=True)
for ax, (name, (xs, ys)) in zip(axes, quartet.items()):
    ax.scatter(xs, ys, color=CORAL, s=45, zorder=3)
    m, c = np.polyfit(xs, ys, 1)
    ax.plot([3, 20], [m * 3 + c, m * 20 + c], color=GRAY, lw=1.5)
    ax.set_title(f"Dataset {name}", loc="left", color=NAVY)
fig.suptitle("Same averages, same correlation, four different stories", x=0.01, ha="left",
             fontweight="bold", color=NAVY, y=1.06)
add_source(fig, "Source: F. J. Anscombe (1973), 'Graphs in Statistical Analysis', The American Statistician")
plt.show()

## Your honest-chart checklist
Before any chart goes on a slide (yours **or** one an AI made), check:
- [ ] Bars start at zero.
- [ ] Time runs left to right, in order, evenly spaced.
- [ ] The time window is wide enough for context, and you can say why it starts where it does.
- [ ] One y-axis per chart (or two panels).
- [ ] Up means more.
- [ ] The title claims only what the data shows (correlation is not cause).
- [ ] The source and any "illustrative" or "estimated" labels are visible.

**Spot it in the wild:** before Thursday, find one misleading chart (news, social media, an ad, a work report). Screenshot it and bring one sentence on what the trick is.